In [2]:
import pandas as pd
import sqlite3

In [3]:
connect = sqlite3.connect('../data/checking-logs.sqlite')
connect

In [18]:
query = """
SELECT checker.uid,
    checker.labname,
    checker.timestamp first_commit_ts,
    pageviews.datetime AS first_view_ts
FROM checker
LEFT JOIN pageviews on checker.uid = pageviews.uid
WHERE status='ready' AND
    numTrials = 1 AND
    labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1') AND
    checker.uid LIKE 'user_%';
"""
datamart = pd.io.sql.read_sql(query, connect, parse_dates=['first_commit_ts', 'first_view_ts'])
datamart

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,NaT
1,user_4,laba04,2020-04-17 11:33:17.366400,NaT
2,user_4,laba04s,2020-04-17 11:48:41.992466,NaT
3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
4,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:05:48.200144
...,...,...,...,...
5659,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-15 13:01:22.753180
5660,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-17 17:39:50.896297
5661,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-18 20:47:36.971195
5662,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-18 22:05:00.106464


In [26]:
test = datamart[datamart['first_view_ts'].notnull()]
control = datamart[datamart['first_view_ts'].isnull()]
test

,uid,labname,first_commit_ts,first_view_ts
3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
4,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:05:48.200144
5,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:06:13.237290
6,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:06:35.574114
7,user_17,project1,2020-04-18 07:56:45.408648,2020-04-21 19:04:25.970852
...,...,...,...,...
5659,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-15 13:01:22.753180
5660,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-17 17:39:50.896297
5661,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-18 20:47:36.971195
5662,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-18 22:05:00.106464


In [28]:
mean_arg = test['first_view_ts'].mean()

In [30]:
control['first_view_ts'] = mean_arg

/var/folders/_d/q8_c5ntn1zsdjh_6zk89_ch40000gn/T/ipykernel_10996/3671784796.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  control['first_view_ts'] = mean_arg


In [31]:
control

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,2020-05-11 21:58:27.011445504
1,user_4,laba04,2020-04-17 11:33:17.366400,2020-05-11 21:58:27.011445504
2,user_4,laba04s,2020-04-17 11:48:41.992466,2020-05-11 21:58:27.011445504
53,user_2,laba04,2020-04-18 13:42:35.482008,2020-05-11 21:58:27.011445504
54,user_2,laba04s,2020-04-18 13:51:22.291271,2020-05-11 21:58:27.011445504
...,...,...,...,...
5091,user_2,laba06s,2020-05-19 14:45:03.908268,2020-05-11 21:58:27.011445504
5401,user_6,laba06s,2020-05-20 14:50:07.609937,2020-05-11 21:58:27.011445504
5551,user_7,laba06s,2020-05-20 23:05:37.742597,2020-05-11 21:58:27.011445504
5552,user_23,laba06,2020-05-21 08:34:10.517205,2020-05-11 21:58:27.011445504


In [32]:
test.to_sql('test', connect)
control.to_sql('control', connect)

81

In [33]:
connect.close()